In [ ]:
import anthropic
import json
import numpy as np
import pandas as pd

client = anthropic.Anthropic()
def get_car_specs(make, model, fuel_type):
    prompt = f"""For the car {make} {model} with fuel type "{fuel_type}", provide the following in strict JSON format only, no other text:
{{
  "cylinders": <number, integer, or null if electric with no engine>,
  "fuel_efficiency_l_per_100km": <number, float, or null if fully electric>,
  "country_of_origin": "<string>"
}}
Consider that the fuel type may indicate a specific trim/variant (e.g. M-performance, Hybrid, Electric) which can have different specs than the base model.
If unsure about a specific value, use null."""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

In [ ]:

results = []

for idx, row in unique_cars.iterrows():
    try:
        specs = get_car_specs(row["Make"], row["Model"], row["fuel_type"])
        specs["Make"] = row["Make"]
        specs["model"] = row["Model"]
        specs["fuel_type"] = row["fuel_type"]
        print(f"Success  for {row['Make']} {row['Model']} ({row['fuel_type']})")
        results.append(specs)
    except Exception as e:
        print(f"Failed for {row['Make']} {row['Model']} ({row['fuel_type']}): {e}")
        results.append({"Make": row["Make"], "Model": row["Model"], "fuel_type": row["fuel_type"],
                        "cylinders": None, "fuel_efficiency_l_per_100km": None,
                        "country_of_origin": None})

specs_df = pd.DataFrame(results)
specs_df.to_csv("DataEnrichment.csv")